# Multi-Agent Deepfake Detection: Data Download & Feature Extraction

This notebook downloads the ASVspoof5 dataset from Kaggle, samples 40% of the files for training and testing, and extracts tabular spectral and prosodic features using multi-processing. The results are saved to Google Drive.

In [ ]:
!pip install kaggle librosa soundfile praat-parselmouth xgboost tqdm pandas numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/40_PER_22_Data')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using Google Drive directory: {DRIVE_DIR}")

## 1. Download Dataset

In [ ]:
import os

# Set your Kaggle credentials here or upload kaggle.json
os.environ['KAGGLE_USERNAME'] = "YOUR_KAGGLE_USERNAME"
os.environ['KAGGLE_KEY'] = "YOUR_KAGGLE_KEY"

DATA_DIR = Path('/content/data/asvspoof5')

if not DATA_DIR.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("Downloading ASVspoof5 from Kaggle...")
    !kaggle datasets download -d aniket202411001/asvspoof5-flac --unzip -p {DATA_DIR}
else:
    print("Dataset already exists.")

## 2. Sample 40% of the Dataset

In [ ]:
import random
import shutil
from tqdm.auto import tqdm

SAMPLE_FRACTION = 0.40
random.seed(42)

splits = ['train', 'test']
classes = ['bonafide', 'spoof']

sampled_files = []

for split in splits:
    for cls in classes:
        # The actual Kaggle paths based on the structure
        src_dir = DATA_DIR / "ASV_Main" / split / cls
        dest_dir = DRIVE_DIR / split / cls
        dest_dir.mkdir(parents=True, exist_ok=True)
        
        if src_dir.exists():
            files = list(src_dir.glob('*.flac'))
            n_sample = int(len(files) * SAMPLE_FRACTION)
            sampled = random.sample(files, n_sample)
            
            print(f"Copying {n_sample} files for {split}/{cls}...")
            for f in tqdm(sampled, leave=False):
                dest_path = dest_dir / f.name
                if not dest_path.exists():
                    shutil.copy2(f, dest_path)
                sampled_files.append((dest_path, 0 if cls == 'bonafide' else 1, split))
        else:
            print(f"Warning: {src_dir} not found!")

print(f"Total sampled files copied to drive: {len(sampled_files)}")

## 3. Extract Features

In [ ]:
# Copy the feature extractor scripts from repo to Colab environment if needed
# Assuming the repo is cloned or scripts are available.
# For now, we will assume spectral_feature_extractor.py and prosodic_feature_extractor.py are in the working directory.

!wget -q https://raw.githubusercontent.com/saltypal/Multi-Agent-Detection-of-AI-Generated-Speech/main/spectral/spectral_feature_extractor.py
!wget -q https://raw.githubusercontent.com/saltypal/Multi-Agent-Detection-of-AI-Generated-Speech/main/prosodic/prosodic_feature_extractor.py

In [ ]:
import pandas as pd
import numpy as np
import concurrent.futures
from spectral_feature_extractor import extract_spectral_row
from prosodic_feature_extractor import extract_prosodic_row

def process_spectral(file_info):
    path, label, split = file_info
    try:
        features = extract_spectral_row(path)
        features['filename'] = path.name
        features['label'] = label
        features['split'] = split
        return features
    except Exception as e:
        print(f"Error processing {path.name}: {e}")
        return None

def process_prosodic(file_info):
    path, label, split = file_info
    try:
        features = extract_prosodic_row(path)
        features['filename'] = path.name
        features['label'] = label
        features['split'] = split
        return features
    except Exception as e:
        print(f"Error processing {path.name}: {e}")
        return None

In [ ]:
def extract_features(extractor_func, output_csv):
    results = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
        futures = [executor.submit(extractor_func, info) for info in sampled_files]
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(sampled_files)):
            res = future.result()
            if res is not None:
                results.append(res)
                
    df = pd.DataFrame(results)
    df.to_csv(DRIVE_DIR / output_csv, index=False)
    print(f"Saved {len(df)} rows to {output_csv}")
    return df

In [ ]:
print("Extracting Spectral Features...")
spectral_df = extract_features(process_spectral, 'spectral_features.csv')
spectral_df.head()

In [ ]:
print("\nExtracting Prosodic Features...")
prosodic_df = extract_features(process_prosodic, 'prosodic_features.csv')
prosodic_df.head()